# CufflessAI: An Educational Blood Pressure Estimation

## Milestone 1 - Data Understanding: Explore dataset, handle missing values, document findings

### Part 1: Data Loading and Inspection

In [ ]:
from pathlib import Path
import mat73 as mat
import pandas as pd
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt

In [ ]:
# Load in data & preprocessing through standard function
def load_and_preprocess_part(part_number: int) -> pd.DataFrame:
    """
    Loads a specified Part dataset (.mat), segments PPG/ABP/ECG into 
    5-second windows (625 samples at 125 Hz), and extracts median SBP/DBP labels.
    """
    notebook_dir = Path.resolve(Path.cwd())
    data_dir = notebook_dir / 'data'
    # Load in one of the part .mat files
    file_path = data_dir / f'Part_{part_number}.mat'
    key = f'Part_{part_number}'
    
    print(f"Loading {file_path.name}...")
    file_data = mat.loadmat(file_path)
    records = file_data[key]
    
    FS = 125 # 125 Hz sampling
    window_size = FS * 5 # 625 samples per signal window, 5-second non-overlapping windows
    window_data = []
    
    for array_idx, record in enumerate(records):
        record = np.asarray(record)
        
        # Make sure shape is (3, x) for PPG, ABP, ECG
        if record.ndim != 2 or record.shape[0] != 3 or record.size == 0:
            continue
            
        ppg, abp, ecg = record[0], record[1], record[2]
        n_points = record.shape[1]
        n_windows = n_points // window_size
        
        for win in range(n_windows):
            begin = win * window_size
            end = begin + window_size
            
            ppg_win = ppg[begin:end]
            abp_win = abp[begin:end]
            ecg_win = ecg[begin:end]
            
            # Skip windows with NaNs or infs
            if np.isnan(ppg_win).any() or np.isinf(ppg_win).any() or \
               np.isnan(abp_win).any() or np.isinf(abp_win).any():
                continue
                
            # Extract systolic and diastolic labels using find_peaks (median method)
            systolic_indices, _ = signal.find_peaks(abp_win)
            diastolic_indices, _ = signal.find_peaks(-abp_win)
            
            # A just-in-case check if peaks aren't found cleanly
            if len(systolic_indices) == 0 or len(diastolic_indices) == 0:
                continue
                
            systolic = np.median(abp_win[systolic_indices])
            diastolic = np.median(abp_win[diastolic_indices])
            
            window_data.append({
                'part': part_number,
                'record_idx': array_idx,
                'window_id': win,
                'ppg': ppg_win,
                'abp': abp_win,
                'ecg': ecg_win,
                'systolic': systolic,
                'diastolic': diastolic,
                'ppg_std': np.std(ppg_win)
            })
            
    df = pd.DataFrame(window_data)
    print(f"Processed Part {part_number}: {len(df)} valid 5-second windows extracted.")
    return df

# Running it on Part 1 file
df_p1 = load_and_preprocess_part(1)

In [ ]:
# Descriptive statistics
metrics_summary = df_p1[['systolic', 'diastolic', 'ppg_std']].describe()
print("--- SUMMARY STATISTICS ---")
print(metrics_summary)

# Skewness
print("\n--- SKEWNESS VALUES ---")
print(df_p1[['systolic', 'diastolic', 'ppg_std']].skew())

In [ ]:
# EDA & Distribution Graphs

# Setting up multi-panel figure for EDA distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) Systolic Blood Pressure (SBP) Distribution
axes[0].hist(df_p1['systolic'], bins=40, color='#edf8b1', edgecolor='black', alpha=0.8)
axes[0].set_title('Distribution of SBP (mmHg)')
axes[0].set_xlabel('Systolic BP')
axes[0].set_ylabel('Frequency')
axes[0].grid(axis='y', alpha=0.3)

# 2) Diastolic Blood Pressure (DBP) Distribution
axes[1].hist(df_p1['diastolic'], bins=40, color='#7fcdbb', edgecolor='black', alpha=0.8)
axes[1].set_title('Distribution of DBP (mmHg)')
axes[1].set_xlabel('Diastolic BP')
axes[1].set_ylabel('Frequency')
axes[1].grid(axis='y', alpha=0.3)

# 3) PPG Standard Deviation Distribution
axes[2].hist(df_p1['ppg_std'], bins=50, color='#2c7fb8', edgecolor='black', alpha=0.8)
axes[2].set_title('Distribution of PPG Window Standard Deviations')
axes[2].set_xlabel('PPG STD')
axes[2].set_ylabel('Frequency')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Calculate dataset-derived percentiles to categorize signal variability (Percentile-based categories)
q1 = df_p1['ppg_std'].quantile(0.25)
q3 = df_p1['ppg_std'].quantile(0.75)

print(f"Dataset 25th Percentile (Q1): {q1:.4f}")
print(f"Dataset 75th Percentile (Q3): {q3:.4f}")

df_p1['ppg_std_category'] = pd.cut(
    df_p1['ppg_std'],
    bins=[-np.inf, q1, q3, np.inf],
    labels=['Low', 'Normal', 'High']
)

print("\n--- PPG STD CATEGORY COUNTS ---")
print(df_p1['ppg_std_category'].value_counts())

In [ ]:
# Visualization of a snapshot of random 5-second windows from Part 1
def plot_percentile_waveforms(df, category_name, num_to_show=3):
    """Plots random 5-second windows from a dataset-derived PPG STD category."""
    subset = df[df['ppg_std_category'] == category_name]
    
    if len(subset) == 0:
        print(f"No windows found in category: {category_name}")
        return
        
    rng = np.random.default_rng(42)
    sample_indices = rng.choice(len(subset), size=min(num_to_show, len(subset)), replace=False)
    
    FS = 125
    window_size = FS * 5
    time = np.arange(window_size) / FS
    
    print(f"\n--- Plotting {category_name} STD Group Samples ---")
    for idx in sample_indices:
        row = subset.iloc[idx]
        rec_idx = row['record_idx']
        ppg_std = row['ppg_std']
        ppg_window = row['ppg']
        
        plt.figure(figsize=(10, 4))
        plt.plot(time, ppg_window, color='#2c7fb8', linewidth=1.5)
        plt.title(f"Part 1 - {category_name} STD PPG\nRecord {rec_idx}, STD = {ppg_std:.3f}")
        plt.xlabel("Time (seconds)")
        plt.ylabel("PPG Amplitude")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

# Visualize sample snapshots from the Low and High percentile categories
plot_percentile_waveforms(df_p1, category_name="Low")
plot_percentile_waveforms(df_p1, category_name="High")